# 17.3 模型进化 (Model Evolution)

模型进化关注模型如何随时间持续改进和适应新需求。

本节涵盖：模型升级、权重插值、能力扩展

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class EvolvableModel(nn.Module):
    def __init__(self, d=64, n_layers=2, n_classes=10):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
        self.head = nn.Linear(d, n_classes)

    def add_layer(self):
        self.layers.append(nn.Linear(self.layers[0].in_features, self.layers[0].in_features))

    def expand_head(self, new_classes=5):
        old_head = self.head
        old_classes = old_head.out_features
        d = old_head.in_features
        new_head = nn.Linear(d, old_classes + new_classes)
        with torch.no_grad():
            new_head.weight[:old_classes] = old_head.weight
            new_head.bias[:old_classes] = old_head.bias
        self.head = new_head

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.head(x)

model = EvolvableModel()
x = torch.randn(4, 64)
out_before = model(x)

model.add_layer()
model.expand_head(new_classes=5)
out_after = model(x)

print('=== Model Evolution ===')
print(f'\nBefore evolution:')
print(f'  Layers: 2, Output classes: 10, Output: {out_before.shape}')
print(f'\nAfter evolution:')
print(f'  Layers: {len(model.layers)}, Output classes: {model.head.out_features}, Output: {out_after.shape}')
print(f'\nEvolution strategies:')
print(f'  1. Layer addition: deeper model for more capacity')
print(f'  2. Head expansion: more classes for new tasks')
print(f'  3. Weight interpolation: merge models trained on different data')
print(f'  4. Knowledge distillation: compress old+new into single model')
print(f'\nKey: Model evolution enables continuous improvement without starting from scratch.')